2. Naloga - chatroom z direktnimi sporoˇcili
Napiˇsite chatroom z direktnimi sporoˇcili. Na streˇznik je prijavljeno poljubno ˇstevilo
uporabnikov, vsak s svojim uporabniˇskim imenom. Cilj aplikacije je, da uporabnik lahko
izbere, kateremu izmed ostalih prijavljenih uporabnikov bo poslal sporoˇcilo. Poslano
sporoˇcilo tako prejme le izbrani uporabnik.
Vsa sporoˇcila naj bodo enkriptirana s hibridno enkripcijo. Naredite par private,
public keys ter session key.
(a) Naloga: Client naj najprej streˇzniku poˇslje svoje uporabniˇsko ime, streˇznik pa si
shrani to ime. Na strani clienta napiˇsite funkcijo trenutni uporabniki, ki izpiˇse vsa
trenutno aktivna uporabniˇska imena.
Namig: Kodo lahko napiˇsete tako, da uporabnik poˇslje v naprej dogovorjeno sporoˇcilo
(npr. ”zahtevaj uporabnike”), streˇznik se pa na to doloˇceno sporoˇcilo odzove tako, da
uporabniku poˇslje seznam vseh aktivnih uporabniˇskih imen
3/4
2. Naloga - chatroom z direktnimi sporoˇcili
(b) Naloga: Dokonˇcajte programa uporabnika in streˇznika, tako da ima uporabnik
moˇznost izvedeti trenutne aktivne uporabnike (tiste, ki jim lahko poˇslje sporoˇcilo) ter
moˇznost poslati sporoˇcilo poljubnemu uporabniku
Navodila za program uporabnika:
• Client najprej streˇzniku poˇslje svoje uporabniˇsko ime
• Nato v neskonˇcni zanki poˇsilja in prejema sporoˇcila. Pri poˇsiljanju sporoˇcil ima
moˇznost zahtevati seznam uporabniˇskih imen (funkcija iz (a) naloge) ali poslati
sporoˇcilo doloˇcenemu uporabniku
• V poslanem sporoˇcilu naj najprej sporoˇci, kateremu uporabniku (glede na
uporabniˇsko ime) je sporoˇcilu namenjeno, nato pa vsebino sporoˇcila
Streˇznik mora skrbeti za sprejemanje novih povezav ter poˇsiljanje sporoˇcil pravemu
uporabniku.

In [ ]:
import socket

while True:
    read_sockets, _, _ = select.select(sockets_list, [], [])

    for notified_socket in read_sockets:

        if notified_socket == server_socket:
            client_socket, client_address = server_socket.accept()

            enc_session_key = client_socket.recv(private_key.size_in_bytes())

            cipher_rsa = PKCS1_OAEP.new(private_key)
            session_key = cipher_rsa.decrypt(enc_session_key)

            session_keys[client_socket] = session_key

            username = receive_encrypted(client_socket, session_key)

            clients[client_socket] = username
            sockets_list.append(client_socket)

            print(f"Nov uporabnik: {username}")

        else:
            try:
                message = receive_encrypted(
                    notified_socket,
                    session_keys[notified_socket]
                )

                sender = clients[notified_socket]

                if message == "LIST":
                    active_users = ", ".join(clients.values())

                    send_encrypted(
                        notified_socket,
                        session_keys[notified_socket],
                        active_users
                    )

                else:
                    receiver, text = message.split(":", 1)

                    target_socket = None

                    for sock, username in clients.items():
                        if username == receiver:
                            target_socket = sock
                            break

                    if target_socket:
                        send_encrypted(
                            target_socket,
                            session_keys[target_socket],
                            f"{sender}: {text}"
                        )

            except:
                print("Odklop:", clients[notified_socket])

                sockets_list.remove(notified_socket)

                del clients[notified_socket]
                del session_keys[notified_socket]

In [ ]:
import socket

IP = "127.0.0.1"
PORT = 6000

public_key = RSA.import_key(open("public.pem").read())

session_key = get_random_bytes(16)

cipher_rsa = PKCS1_OAEP.new(public_key)
enc_session_key = cipher_rsa.encrypt(session_key)

client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client_socket.connect((IP, PORT))
client_socket.setblocking(False)

client_socket.sendall(enc_session_key)



def send_encrypted(message):
    cipher = AES.new(session_key, AES.MODE_EAX)

    ciphertext, tag = cipher.encrypt_and_digest(message.encode())

    client_socket.sendall(cipher.nonce)
    client_socket.sendall(tag)
    client_socket.sendall(ciphertext)



def receive_encrypted():
    nonce = client_socket.recv(16)
    tag = client_socket.recv(16)
    ciphertext = client_socket.recv(1024)

    cipher = AES.new(session_key, AES.MODE_EAX, nonce)

    data = cipher.decrypt_and_verify(ciphertext, tag)

    return data.decode()


username = input("Username: ")
send_encrypted(username)

print("Ukazi:")
print("LIST")
print("uporabnik:sporočilo")

while True:
    message = input(f"{username}> ")

    send_encrypted(message)

    try:
        incoming = receive_encrypted()
        print("Prejeto:", incoming)
    except:
        pass